In [1]:
import os
import re
import json
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [2]:
SEED = 42

DATASET_ROOT = Path("/kaggle/input/datasets/afenmarbun/wilddeepfake-46f")
OUTPUT_DIR = Path("/kaggle/working/wdf46f_index")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.20
SOURCE_CLIP_LEN = 46
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

CLASS_TO_IDX = {"real": 0, "fake": 1}

random.seed(SEED)
np.random.seed(SEED)

print("DATASET_ROOT :", DATASET_ROOT)
print("OUTPUT_DIR   :", OUTPUT_DIR)

DATASET_ROOT : /kaggle/input/datasets/afenmarbun/wilddeepfake-46f
OUTPUT_DIR   : /kaggle/working/wdf46f_index


In [3]:
@dataclass(frozen=True)
class ExperimentConfig:
    code: str
    description: str
    num_frames: int
    stride: int
    final_image_size: int = 224
    downscale_first: int | None = None


EXPERIMENTS = {
    "E01": ExperimentConfig("E01", "Konfigurasi dasar", 16, 3, 224, None),
    "E02": ExperimentConfig("E02", "Penurunan resolusi 50%", 16, 3, 224, 112),
    "E03": ExperimentConfig("E03", "Variasi stride menengah", 16, 2, 224, None),
    "E04": ExperimentConfig("E04", "Variasi stride sangat rapat", 16, 1, 224, None),
    "E05": ExperimentConfig("E05", "Variasi jumlah frame lebih pendek", 8, 3, 224, None),
    "E06": ExperimentConfig("E06", "Variasi jumlah frame menengah di bawah 16", 12, 3, 224, None),
    "E07": ExperimentConfig("E07", "Variasi jumlah frame di atas 16", 20, 2, 224, None),
    "E08": ExperimentConfig("E08", "Variasi jumlah frame lebih panjang", 24, 1, 224, None),
}

In [4]:
def numeric_key(path: Path):
    nums = re.findall(r"\d+", path.stem)
    return int(nums[-1]) if nums else path.stem


def list_image_files(folder: Path):
    return sorted(
        [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS],
        key=numeric_key
    )


def find_subset_dirs(dataset_root: Path):
    subset_dirs = {}

    for root, dirs, files in os.walk(dataset_root):
        for d in dirs:
            if d in {"real_train", "real_test", "fake_train", "fake_test"}:
                subset_dirs[d] = Path(root) / d

    required = {"real_train", "real_test", "fake_train", "fake_test"}
    missing = required - set(subset_dirs.keys())

    if missing:
        raise FileNotFoundError(
            f"Folder subset berikut tidak ditemukan: {sorted(missing)}\n"
            f"Periksa kembali struktur dataset di: {dataset_root}"
        )

    return subset_dirs


def collect_clip_dirs(subset_dir: Path):
    clip_dirs = []

    for root, dirs, files in os.walk(subset_dir):
        image_files = [f for f in files if Path(f).suffix.lower() in IMAGE_EXTS]
        if image_files:
            clip_dirs.append(Path(root))

    return sorted(clip_dirs)

In [5]:
subset_dirs = find_subset_dirs(DATASET_ROOT)

print("Subset ditemukan:")
for k, v in subset_dirs.items():
    print(f"- {k:10s}: {v}")


def extract_source_clip_name(folder_name: str) -> str:
    """
    Mengambil ID klip asal dari nama folder turunan.
    Contoh:
    - 101_0 -> 101
    - 101_1 -> 101
    - abc_12 -> abc
    Jika tidak cocok pola *_angka, nama folder dikembalikan apa adanya.
    """
    m = re.match(r"^(.*)_(\d+)$", folder_name)
    return m.group(1) if m else folder_name


def build_manifest_for_subset(subset_name: str, subset_dir: Path):
    label_name = "real" if subset_name.startswith("real") else "fake"
    label = CLASS_TO_IDX[label_name]

    records = []
    for clip_dir in collect_clip_dirs(subset_dir):
        frame_files = list_image_files(clip_dir)

        clip_dir_rel = clip_dir.relative_to(DATASET_ROOT)
        clip_folder_name = clip_dir.name
        source_clip_name = extract_source_clip_name(clip_folder_name)

        # Parent relatif terhadap subset, misalnya:
        # fake_train/1/101_0  -> parent_in_subset = "1"
        # real_train/0/55_2   -> parent_in_subset = "0"
        parent_in_subset = str(clip_dir.relative_to(subset_dir).parent)

        # Group key untuk mencegah leakage.
        # Semua turunan dari klip asal yang sama akan memiliki group_id yang sama.
        group_id = f"{subset_name}/{parent_in_subset}/{source_clip_name}"

        records.append({
            "subset": subset_name,
            "split": "train" if subset_name.endswith("train") else "test",
            "label": label,
            "label_name": label_name,
            "clip_dir_abs": str(clip_dir),
            "clip_dir_rel": str(clip_dir_rel),
            "clip_folder_name": clip_folder_name,
            "source_clip_name": source_clip_name,
            "group_parent": parent_in_subset,
            "group_id": group_id,
            "num_frames": len(frame_files),
        })

    return records


train_records = []
train_records += build_manifest_for_subset("real_train", subset_dirs["real_train"])
train_records += build_manifest_for_subset("fake_train", subset_dirs["fake_train"])

test_records = []
test_records += build_manifest_for_subset("real_test", subset_dirs["real_test"])
test_records += build_manifest_for_subset("fake_test", subset_dirs["fake_test"])

train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)

print("Jumlah klip train resmi:", len(train_df))
print("Jumlah klip test resmi :", len(test_df))

print("\nJumlah group train unik:", train_df["group_id"].nunique())
print("Jumlah group test unik :", test_df["group_id"].nunique())

display(train_df.head())

Subset ditemukan:
- real_train: /kaggle/input/datasets/afenmarbun/wilddeepfake-46f/wilddeepfake_46f_merged_v5/real_train
- real_test : /kaggle/input/datasets/afenmarbun/wilddeepfake-46f/wilddeepfake_46f_merged_v5/real_test
- fake_test : /kaggle/input/datasets/afenmarbun/wilddeepfake-46f/archive/fake_test
- fake_train: /kaggle/input/datasets/afenmarbun/wilddeepfake-46f/archive/fake_train
Jumlah klip train resmi: 19071
Jumlah klip test resmi : 3232

Jumlah group train unik: 6508
Jumlah group test unik : 806


,subset,split,label,label_name,clip_dir_abs,clip_dir_rel,clip_folder_name,source_clip_name,group_parent,group_id,num_frames
0,real_train,train,0,real,/kaggle/input/datasets/afenmarbun/wilddeepfake...,wilddeepfake_46f_merged_v5/real_train/1/103_0,103_0,103,1,real_train/1/103,46
1,real_train,train,0,real,/kaggle/input/datasets/afenmarbun/wilddeepfake...,wilddeepfake_46f_merged_v5/real_train/1/175_0,175_0,175,1,real_train/1/175,46
2,real_train,train,0,real,/kaggle/input/datasets/afenmarbun/wilddeepfake...,wilddeepfake_46f_merged_v5/real_train/1/318_0,318_0,318,1,real_train/1/318,46
3,real_train,train,0,real,/kaggle/input/datasets/afenmarbun/wilddeepfake...,wilddeepfake_46f_merged_v5/real_train/1/506_0,506_0,506,1,real_train/1/506,46
4,real_train,train,0,real,/kaggle/input/datasets/afenmarbun/wilddeepfake...,wilddeepfake_46f_merged_v5/real_train/1/619_0,619_0,619,1,real_train/1/619,46


In [6]:
def validate_clip_length(dataframe: pd.DataFrame, expected_len: int = 46):
    invalid = dataframe[dataframe["num_frames"] != expected_len].copy()

    if len(invalid) > 0:
        display(invalid.head(20))
        raise ValueError(
            f"Ditemukan {len(invalid)} folder yang tidak berisi {expected_len} frame."
        )

    print(f"Semua klip valid: {len(dataframe)} folder memiliki {expected_len} frame.")


validate_clip_length(train_df, expected_len=SOURCE_CLIP_LEN)
validate_clip_length(test_df, expected_len=SOURCE_CLIP_LEN)

Semua klip valid: 19071 folder memiliki 46 frame.
Semua klip valid: 3232 folder memiliki 46 frame.


In [7]:
# Split dilakukan pada level GROUP, bukan level klip,
# agar folder turunan seperti 101_0, 101_1, 101_2 tidak terpisah.

group_df = (
    train_df[["group_id", "label", "label_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

train_groups, val_groups = train_test_split(
    group_df,
    test_size=VAL_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=group_df["label"],
)

train_group_ids = set(train_groups["group_id"])
val_group_ids = set(val_groups["group_id"])

train_split_df = (
    train_df[train_df["group_id"].isin(train_group_ids)]
    .copy()
    .reset_index(drop=True)
)

val_split_df = (
    train_df[train_df["group_id"].isin(val_group_ids)]
    .copy()
    .reset_index(drop=True)
)

test_df = test_df.reset_index(drop=True)

# Validasi anti-leakage
overlap_groups = set(train_split_df["group_id"]) & set(val_split_df["group_id"])
if overlap_groups:
    raise ValueError(f"Ditemukan kebocoran group antara train dan validasi: {list(overlap_groups)[:5]}")

print("Jumlah group train final   :", train_split_df["group_id"].nunique())
print("Jumlah group validasi final:", val_split_df["group_id"].nunique())
print("Jumlah group test final    :", test_df["group_id"].nunique())

print("\nJumlah klip train final   :", len(train_split_df))
print("Jumlah klip validasi final:", len(val_split_df))
print("Jumlah klip test final    :", len(test_df))

print("\nDistribusi label train (klip):")
print(train_split_df["label_name"].value_counts())

print("\nDistribusi label validasi (klip):")
print(val_split_df["label_name"].value_counts())

print("\nDistribusi label test (klip):")
print(test_df["label_name"].value_counts())

print("\nDistribusi label train (group):")
print(train_groups["label_name"].value_counts())

print("\nDistribusi label validasi (group):")
print(val_groups["label_name"].value_counts())

Jumlah group train final   : 5206
Jumlah group validasi final: 1302
Jumlah group test final    : 806

Jumlah klip train final   : 15500
Jumlah klip validasi final: 3571
Jumlah klip test final    : 3232

Distribusi label train (klip):
label_name
fake    10124
real     5376
Name: count, dtype: int64

Distribusi label validasi (klip):
label_name
fake    2189
real    1382
Name: count, dtype: int64

Distribusi label test (klip):
label_name
fake    2139
real    1093
Name: count, dtype: int64

Distribusi label train (group):
label_name
real    2727
fake    2479
Name: count, dtype: int64

Distribusi label validasi (group):
label_name
real    682
fake    620
Name: count, dtype: int64


In [8]:
def compute_temporal_sampling_plan(total_frames: int, num_frames: int, stride: int):
    required_length = (num_frames - 1) * stride + 1
    if required_length > total_frames:
        raise ValueError(
            f"Konfigurasi tidak valid: T={num_frames}, stride={stride}, "
            f"L={required_length}, N={total_frames}"
        )

    remaining = total_frames - required_length
    discard_left = remaining // 2
    discard_right = remaining - discard_left

    start = discard_left
    sampled_indices = [start + k * stride for k in range(num_frames)]

    return {
        "N": total_frames,
        "T": num_frames,
        "tau": stride,
        "L": required_length,
        "R": remaining,
        "discard_left": discard_left,
        "discard_right": discard_right,
        "sampled_indices_0based": sampled_indices,
        "sampled_indices_1based": [x + 1 for x in sampled_indices],
    }


sampling_preview = []
for code, cfg in EXPERIMENTS.items():
    plan = compute_temporal_sampling_plan(
        total_frames=SOURCE_CLIP_LEN,
        num_frames=cfg.num_frames,
        stride=cfg.stride,
    )
    sampling_preview.append({
        "code": code,
        "description": cfg.description,
        "num_frames": cfg.num_frames,
        "stride": cfg.stride,
        "final_image_size": cfg.final_image_size,
        "downscale_first": cfg.downscale_first,
        **plan
    })

sampling_preview_df = pd.DataFrame(sampling_preview)

train_df.to_csv(OUTPUT_DIR / "manifest_train.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "manifest_test.csv", index=False)
train_split_df.to_csv(OUTPUT_DIR / "train_split.csv", index=False)
val_split_df.to_csv(OUTPUT_DIR / "val_split.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_split.csv", index=False)
sampling_preview_df.to_csv(OUTPUT_DIR / "sampling_preview.csv", index=False)

dataset_info = {
    "dataset_root": str(DATASET_ROOT),
    "source_clip_len": SOURCE_CLIP_LEN,
    "val_ratio": VAL_RATIO,
    "seed": SEED,
    "class_to_idx": CLASS_TO_IDX,
    "subset_dirs": {k: str(v) for k, v in subset_dirs.items()},
}

with open(OUTPUT_DIR / "dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

with open(OUTPUT_DIR / "experiment_configs.json", "w") as f:
    json.dump({k: asdict(v) for k, v in EXPERIMENTS.items()}, f, indent=2)

print("Semua file metadata berhasil disimpan.")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p)

Semua file metadata berhasil disimpan.
- /kaggle/working/wdf46f_index/dataset_info.json
- /kaggle/working/wdf46f_index/experiment_configs.json
- /kaggle/working/wdf46f_index/manifest_test.csv
- /kaggle/working/wdf46f_index/manifest_train.csv
- /kaggle/working/wdf46f_index/sampling_preview.csv
- /kaggle/working/wdf46f_index/test_split.csv
- /kaggle/working/wdf46f_index/train_split.csv
- /kaggle/working/wdf46f_index/val_split.csv
